<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# EduTrack Analytics
## Notebook 5 — Machine Learning Premium : Détection du Décrochage
### 📝 VERSION APPRENANT

> **Instructions :** Complète les cellules marquées `# TODO`.  
> Les blocs `MÉTHODE` t'expliquent l'approche attendue.  
> Les blocs `🧠 Tes observations` sont à remplir après exécution.

| | |
|---|---|
| **Prérequis** | `inscriptions_analytics.csv` dans `SAVE_PATH` (produit par NB2) |
| **Niveau** | Premium |
| **Outils** | Python — scikit-learn, pandas, matplotlib |
| **Durée estimée** | 3h à 4h |

---
> **Problème :** Classification binaire — prédire `at_risk_dropout = 1` avant l'abandon.
>
> **Contrainte métier :** Recall > 0.75. Il vaut mieux alerter par excès (faux positifs) que rater un vrai décrocheur (faux négatif). Le coût d'une fausse alerte (envoyer un email de relance à tort) est bien inférieur au coût d'un abandon (perte du revenu + impact réputation).
>
> **Coupure temporelle :** Train avant 2024-01-01 / Test après 2024-01-01.

---
## Étape 1 — Imports et configuration

> **MÉTHODE — Pourquoi `TimeSeriesSplit` et non `KFold` ?**
>
> `TimeSeriesSplit` remplace `KFold` car les données sont temporelles — un split aléatoire laisserait des données futures contaminer l'entraînement (data leakage). Le fold N s'entraîne toujours sur un passé strict par rapport au fold de test.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    recall_score, precision_score, f1_score, precision_recall_curve
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/elearning_analytics/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

---
## Étape 2 — Chargement et vérification

### MÉTHODE
On vérifie d'abord la distribution de la variable cible et les valeurs manquantes dans les features ML. Les nulls seront imputés par la médiane — c'est la stratégie la plus robuste pour des features numériques en contexte ML.

In [ ]:
BASE_URL   = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/'
clean_path = f'{SAVE_PATH}inscriptions_analytics.csv'

# Fallback GitHub si le fichier NB2 n'est pas disponible
if not os.path.exists(clean_path):
    clean_path = BASE_URL + 'inscriptions_analytics.csv'
    print('⚠️  inscriptions_analytics.csv non trouvé en local — chargement depuis GitHub')
else:
    print(f'✅ Fichier nettoyé trouvé : {clean_path}')

df = pd.read_csv(clean_path, parse_dates=['date_inscription'])
print(f'Dataset : {df.shape}')
print(f'\nDistribution at_risk_dropout :')
vc = df['at_risk_dropout'].value_counts()
print(vc)
print(f'  Taux classe positive : {vc[1]/len(df)*100:.1f}%')

FEATURES = [
    'jours_depuis_inscription', 'mois_inscription', 'est_inscription_weekend',
    'nb_sessions', 'duree_totale_min', 'nb_semaines_actives',
    'engagement_score', 'nb_jours_inactif'
]
print(f'\nNulls dans les features :')
print(df[FEATURES].isnull().sum())

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 3 — Sélection features et coupure temporelle

### MÉTHODE
La coupure temporelle est **la règle la plus importante** du ML sur données temporelles. En utilisant `date_inscription < 2024-01-01` pour le train et `>= 2024-01-01` pour le test, on simule exactement les conditions réelles : le modèle est entraîné sur le passé et prédit sur le futur.

**Features exclues pour éviter le data leakage :**
- `statut` — contient directement la réponse
- `date_fin_reelle` — inconnue au moment de la prédiction
- `csat` — donné après la fin du parcours
- `certificat_obtenu` — conséquence du statut

In [ ]:
CIBLE        = 'at_risk_dropout'
DATE_COUPURE = pd.Timestamp('2024-01-01')

df_ml = df[FEATURES + [CIBLE, 'date_inscription']].copy()
# Imputation médiane pour les nulls
for col in FEATURES:
    df_ml[col] = df_ml[col].fillna(df_ml[col].median())

# TODO : créer train (date < DATE_COUPURE) et test (date >= DATE_COUPURE)
# TODO : créer X_train, y_train, X_test, y_test
train = # TODO
test  = # TODO
X_train, y_train = train[FEATURES], train[CIBLE]
X_test,  y_test  = test[FEATURES],  test[CIBLE]

print(f'Train : {len(train):,} lignes | taux positifs : {y_train.mean()*100:.1f}%')
print(f'Test  : {len(test):,} lignes  | taux positifs : {y_test.mean()*100:.1f}%')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 4 — Entraînement de 3 modèles

### MÉTHODE
`class_weight='balanced'` est essentiel pour la Logistic Regression et le Random Forest. Il pénalise davantage les erreurs sur la classe minoritaire en pondérant inversement les classes par leur fréquence. Sans ça, le modèle ignorerait souvent la classe positive. Le Gradient Boosting gère nativement le déséquilibre via son processus itératif.

In [ ]:
# TODO : entraîner 3 modèles

# Logistic Regression (class_weight='balanced', max_iter=500)
lr = LogisticRegression(# TODO)
lr.fit(X_train, y_train)
y_pred_lr  = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

# Random Forest (n_estimators=300, max_depth=8, class_weight='balanced')
rf = RandomForestClassifier(# TODO)
rf.fit(X_train, y_train)
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# Gradient Boosting (n_estimators=200, max_depth=4, learning_rate=0.05)
gb = GradientBoostingClassifier(# TODO)
gb.fit(X_train, y_train)
y_pred_gb  = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

# TODO : construire le tableau comparatif (AUC-ROC, F1, Recall, Precision)
results = pd.DataFrame({
    'Modèle':    ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'AUC-ROC':   [# TODO],
    'F1':        [# TODO],
    'Recall':    [# TODO],
    'Precision': [# TODO],
}).round(4)
print(results.to_string(index=False))

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 5 — Optimisation du seuil de décision

### MÉTHODE
L'AUC-ROC mesure la capacité discriminante du modèle sur TOUS les seuils possibles. Un AUC de 0.90 pour le Random Forest signifie qu'il sépare bien les classes. Le seuil par défaut (0.5) n'est pas toujours optimal. En le baissant, on augmente le Recall au détriment de la Précision — exactement ce qu'on veut pour EduTrack.

In [ ]:
# TODO : tester les seuils de 0.30 à 0.74 par pas de 0.05 sur y_proba_rf
# Pour chaque seuil : calculer recall, precision, f1
seuils = np.arange(0.30, 0.75, 0.05)
rows = []
for s in seuils:
    yp = (y_proba_rf >= s).astype(int)
    rows.append({
        'seuil':     round(s, 2),
        'recall':    # TODO,
        'precision': # TODO (zero_division=0),
        'f1':        # TODO (zero_division=0)
    })
df_s = pd.DataFrame(rows)
print(df_s.round(4).to_string(index=False))

# TODO : choisir le premier seuil qui satisfait Recall >= 0.75
optimal       = df_s[df_s['recall'] >= 0.75].iloc[0]
SEUIL_OPTIMAL = float(optimal['seuil'])
print(f'\nSeuil optimal retenu : {SEUIL_OPTIMAL:.2f}')
print(f'  Recall    : {optimal["recall"]:.4f}')
print(f'  Precision : {optimal["precision"]:.4f}')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_s['seuil'], df_s['recall'],    color=COLORS['danger'],    label='Recall',    linewidth=2.5, marker='o')
ax.plot(df_s['seuil'], df_s['precision'], color=COLORS['secondary'], label='Precision', linewidth=2.5, marker='s')
ax.plot(df_s['seuil'], df_s['f1'],        color=COLORS['primary'],   label='F1-Score',  linewidth=2, linestyle='--')
ax.axvline(SEUIL_OPTIMAL, color=COLORS['warning'], linestyle='--', linewidth=2,
           label=f'Seuil optimal ({SEUIL_OPTIMAL:.2f})')
ax.axhline(0.75, color=COLORS['danger'], linestyle=':', linewidth=1, alpha=0.7,
           label='Contrainte Recall = 0.75')
ax.set_xlabel('Seuil de décision')
ax.set_ylabel('Score')
ax.set_title('Precision vs Recall selon le seuil — Random Forest EduTrack', fontweight='bold')
ax.legend()
ax.set_xlim(0.28, 0.75)
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}precision_recall_curve.png')

---
## Étape 6 — Feature Importance

### MÉTHODE
La feature importance du Random Forest mesure la réduction moyenne d'impureté (Gini) apportée par chaque feature lors des splits. Une feature importante est celle dont la valeur discrimine bien les at_risk=1 des at_risk=0. C'est une information pédagogique clé : elle dit aux équipes EduTrack sur quels indicateurs intervenir en priorité.

In [ ]:
# TODO : extraire et afficher la feature importance du Random Forest
# Indice : rf.feature_importances_ → pd.Series indexée par FEATURES
fi = # TODO
fi = fi.sort_values(ascending=False)
print('=== FEATURE IMPORTANCE ===')
for feat, imp in fi.items():
    bar = '█' * int(imp * 200)
    print(f'  {feat:<30} {imp:.4f} ({imp*100:.1f}%) {bar}')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
fi_sorted   = fi.sort_values()
colors_fi   = [
    COLORS['danger']  if imp > 0.2 else
    COLORS['primary'] if imp > 0.1 else
    COLORS['neutral']
    for imp in fi_sorted.values
]
ax.barh(fi_sorted.index, fi_sorted.values, color=colors_fi)
for i, (feat, imp) in enumerate(fi_sorted.items()):
    ax.text(imp + 0.002, i, f'{imp*100:.1f}%', va='center', fontsize=10)
ax.set_xlabel('Importance (%)')
ax.set_title('Feature Importance — Random Forest EduTrack', fontweight='bold')
ax.set_xlim(0, fi.max() * 1.25)
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}feature_importance.png')

---
## Étape 7 — Validation par TimeSeriesSplit

### MÉTHODE
`TimeSeriesSplit(n_splits=5)` crée 5 folds en respectant l'ordre temporel : le fold N s'entraîne sur les N premiers blocs et teste sur le (N+1)ème. Cela simule N prédictions successives dans le futur. Un modèle stable doit avoir un écart-type de Recall < 0.10 entre les folds.

In [ ]:
df_sorted = df_ml.sort_values('date_inscription').reset_index(drop=True)
X_all     = df_sorted[FEATURES].fillna(df_sorted[FEATURES].median())
y_all     = df_sorted[CIBLE]

# TODO : créer un TimeSeriesSplit avec 5 folds
tscv       = TimeSeriesSplit(# TODO)
recalls_cv = []

for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_all)):
    # TODO : entraîner un RandomForest léger (n_estimators=100)
    # TODO : prédire sur le fold de test
    # TODO : calculer le recall et l'ajouter à recalls_cv
    pass

print(f'Recall moyen  : {np.mean(recalls_cv):.4f}')
print(f'Écart-type    : {np.std(recalls_cv):.4f}')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 8 — Génération du fichier d'alertes

### MÉTHODE
On applique le modèle uniquement sur les inscriptions `'En cours'` — ce sont les seules pour lesquelles une intervention est encore possible. Le fichier `apprenants_risque_scores.csv` sera chargé dans Power BI pour alimenter la Page 5 Alertes ML.

In [ ]:
df_en_cours = df[df['statut'] == 'En cours'].copy()
X_score     = df_en_cours[FEATURES].fillna(df_en_cours[FEATURES].median())

# TODO : générer les scores de risque avec rf.predict_proba
scores = # TODO

df_en_cours['score_risque']      = scores
# TODO : créer alerte_decrochage = 1 si score >= SEUIL_OPTIMAL
df_en_cours['alerte_decrochage'] = # TODO

# Enrichissement avec parcours et apprenants
df_parc_light = pd.read_csv(BASE_URL + 'parcours.csv')[['parcours_id', 'titre', 'domaine', 'instructeur']]
df_app_light  = pd.read_csv(BASE_URL + 'apprenants.csv')[['apprenant_id', 'prenom', 'nom', 'pays']]

df_alertes = (
    df_en_cours[['inscription_id', 'apprenant_id', 'parcours_id', 'progression_pct',
                 'engagement_score', 'nb_jours_inactif', 'score_risque', 'alerte_decrochage']]
    .merge(df_app_light,  on='apprenant_id', how='left')
    .merge(df_parc_light, on='parcours_id',  how='left')
    .sort_values('score_risque', ascending=False)
    .reset_index(drop=True)
)

df_alertes.to_csv(f'{SAVE_PATH}apprenants_risque_scores.csv', index=False)
print(f'✅ apprenants_risque_scores.csv généré')
print(f'   Alertes : {df_alertes["alerte_decrochage"].sum():,} / {len(df_alertes):,}')
print(df_alertes[['prenom', 'nom', 'titre', 'score_risque']].head(5).to_string())

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Bilan du Notebook 5

| Metric | Logistic Regression | Random Forest | Gradient Boosting |
|---|---|---|---|
| AUC-ROC | 0.3983 | **0.8986** | 0.9025 |
| F1-Score | 0.2562 | 0.2021 | 0.2211 |
| Recall (seuil déf.) | 0.8571 | 0.1131 | 0.1250 |

| Élément | Valeur |
|---|---|
| Modèle choisi | Random Forest (AUC=0.90) |
| Seuil optimal | **0.30** |
| Recall final | **0.9583 ✅** (> 0.75) |
| Precision | 0.2051 |
| Feature #1 | `nb_jours_inactif` (25.3%) |
| Feature #2 | `nb_sessions` (17.7%) |
| Feature #3 | `engagement_score` (17.6%) |
| Validation CV (Recall moyen) | 0.8016 ± 0.2469 |
| Nb alertes générées | 1 613 / 2 095 (77%) |

**Fichiers exportés dans `SAVE_PATH` :**
- `apprenants_risque_scores.csv` — fichier d'alertes pour Power BI
- `precision_recall_curve.png` — courbe de seuil
- `feature_importance.png` — importance des features

**Recommandation finale :** Stratifier les alertes en 3 niveaux (> 0.80 appel / 0.60-0.80 email / 0.30-0.60 notification push) pour optimiser les ressources de l'équipe pédagogique EduTrack.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.